# Train Earbud Left/Right — Kaggle Run All

Dùng cho output `datasets/earbud_rnn_merged`. Add Input, bật GPU + Internet, sửa duy nhất `DATASET_ROOT` trong CELL 1 rồi bấm **Run All**.

In [ ]:
# CELL 1 — CHỈ SỬA DÒNG DATASET_ROOT
DATASET_ROOT = r'/kaggle/input/CHANGE_ME/earbud_rnn_merged'

MODEL_WEIGHTS = 'yolo11s.pt'
EPOCHS = 80
IMAGE_SIZE = 640
BATCH_SIZE = 16
PATIENCE = 20
RUN_NAME = 'earbud_lr_detector'

In [ ]:
# CELL 2 — Cài thư viện
%pip install -q -U ultralytics

In [ ]:
# CELL 3 — Kiểm tra GPU, class, ảnh và label
import json
import shutil
from collections import Counter
from pathlib import Path

import torch
import yaml

EXPECTED_CLASSES = [
    'open_case', 'close_case', 'left_earbud', 'right_earbud',
    'empty_left', 'empty_right', 'hand'
]
DATASET_ROOT = Path(DATASET_ROOT)
assert DATASET_ROOT.is_dir(), f'Không tìm thấy: {DATASET_ROOT}'
assert torch.cuda.is_available(), 'Hãy bật Settings → Accelerator → GPU rồi Run All lại.'
source_yaml = DATASET_ROOT / 'data.yaml'
assert source_yaml.is_file(), f'Không tìm thấy {source_yaml}'
with source_yaml.open(encoding='utf-8') as file:
    source_config = yaml.safe_load(file)
source_names = source_config.get('names', [])
if isinstance(source_names, dict):
    source_names = [source_names[index] for index in sorted(source_names)]
assert source_names == EXPECTED_CLASSES, (
    f'Class không đúng. Hiện có: {source_names}; cần: {EXPECTED_CLASSES}'
)

split_folders = {'train': 'train', 'val': 'valid', 'test': 'test'}
report = {}
class_counts = Counter()
for split, folder in split_folders.items():
    image_dir = DATASET_ROOT / folder / 'images'
    label_dir = DATASET_ROOT / folder / 'labels'
    assert image_dir.is_dir() and label_dir.is_dir(), f'Thiếu {folder}/images hoặc labels'
    images = [path for path in image_dir.iterdir() if path.is_file()]
    labels = list(label_dir.glob('*.txt'))
    assert len(images) == len(labels), f'{split}: {len(images)} ảnh != {len(labels)} labels'
    boxes = 0
    for label in labels:
        for line_number, row in enumerate(label.read_text().splitlines(), start=1):
            if not row.strip():
                continue
            fields = row.split()
            assert len(fields) == 5, f'{label}:{line_number}: label không phải bbox YOLO'
            class_id = int(fields[0])
            values = [float(value) for value in fields[1:]]
            assert 0 <= class_id < len(EXPECTED_CLASSES), f'{label}: class_id={class_id}'
            assert all(0 <= value <= 1 for value in values), f'{label}: tọa độ ngoài [0,1]'
            assert values[2] > 0 and values[3] > 0, f'{label}: box không có diện tích'
            class_counts[class_id] += 1
            boxes += 1
    report[split] = {'images': len(images), 'boxes': boxes}

missing = [EXPECTED_CLASSES[i] for i in range(len(EXPECTED_CLASSES)) if class_counts[i] == 0]
assert not missing, f'Class không có box: {missing}'
ABSOLUTE_YAML = Path('/kaggle/working/data_earbud_lr.yaml')
ABSOLUTE_YAML.write_text(
    f'path: {DATASET_ROOT.as_posix()}\n'
    'train: train/images\n'
    'val: valid/images\n'
    'test: test/images\n\n'
    f'nc: {len(EXPECTED_CLASSES)}\n'
    f'names: {json.dumps(EXPECTED_CLASSES)}\n',
    encoding='utf-8',
)
print('GPU:', torch.cuda.get_device_name(0))
print('Splits:', report)
for class_id, name in enumerate(EXPECTED_CLASSES):
    print(f'{class_id}: {name} = {class_counts[class_id]} boxes')
print('data.yaml:', ABSOLUTE_YAML)

In [ ]:
# CELL 4 — Train YOLO11s
from ultralytics import YOLO

TRAINING_ROOT = Path('/kaggle/working/training')
RUN_DIR = TRAINING_ROOT / RUN_NAME
if RUN_DIR.exists():
    shutil.rmtree(RUN_DIR)
local_weights = list(DATASET_ROOT.rglob(MODEL_WEIGHTS))
model_source = str(local_weights[0]) if local_weights else MODEL_WEIGHTS
model = YOLO(model_source)
model.train(
    data=str(ABSOLUTE_YAML), epochs=EPOCHS, imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE, device=0, workers=2, project=str(TRAINING_ROOT),
    name=RUN_NAME, exist_ok=True, pretrained=True, optimizer='auto',
    patience=PATIENCE, seed=42, deterministic=True, cache='disk',
    amp=True, plots=True, save=True, save_period=10, verbose=True,
)

In [ ]:
# CELL 5 — Đánh giá best.pt trên test
BEST_PATH = RUN_DIR / 'weights' / 'best.pt'
assert BEST_PATH.is_file(), f'Không tìm thấy {BEST_PATH}'
best_model = YOLO(str(BEST_PATH))
metrics = best_model.val(
    data=str(ABSOLUTE_YAML), split='test', imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE, device=0, plots=True,
    project=str(TRAINING_ROOT), name=f'{RUN_NAME}_test',
)
print(f'mAP50: {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')
print(f'Precision: {metrics.box.mp:.4f}')
print(f'Recall: {metrics.box.mr:.4f}')

In [ ]:
# CELL 6 — Đóng gói kết quả
PORTABLE_BEST = Path('/kaggle/working/best_earbud_lr_detector.pt')
shutil.copy2(BEST_PATH, PORTABLE_BEST)
shutil.copy2(ABSOLUTE_YAML, RUN_DIR / 'data_used.yaml')
summary = {
    'classes': EXPECTED_CLASSES, 'dataset': report,
    'map50': float(metrics.box.map50), 'map50_95': float(metrics.box.map),
    'precision': float(metrics.box.mp), 'recall': float(metrics.box.mr),
}
(RUN_DIR / 'training_summary.json').write_text(
    json.dumps(summary, indent=2) + '\n', encoding='utf-8'
)
archive = shutil.make_archive(
    '/kaggle/working/earbud_lr_training_results', 'zip', root_dir=RUN_DIR
)
print('TRAIN HOÀN TẤT — tải trong Output:')
print(PORTABLE_BEST)
print(archive)

In [ ]:
# CELL 7 — Hiển thị biểu đồ
from IPython.display import Image, display
for name in ('results.png', 'confusion_matrix_normalized.png', 'PR_curve.png'):
    path = RUN_DIR / name
    if path.is_file():
        print(name)
        display(Image(filename=str(path)))